# Notebook 06: Final Visualizations & Results

Generates all 10 key figures and LaTeX tables for the thesis.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14

from pathlib import Path
from src.data_collection import load_processed
from src.data_processing import PROCESSED_DIR
from src.dls_method import DLSModel
from src.feature_engineering import prepare_features, load_prepared_data
from src.ml_models import load_model, predict_with_model
from src.evaluation import (
    compute_metrics, compare_models, full_evaluation,
    generate_latex_table, cross_format_heatmap_data,
    TABLES_DIR, FIGURES_DIR
)
from src.visualizations import (
    plot_dls_resource_curves, plot_model_comparison_bar,
    plot_error_distributions, plot_phase_rmse,
    plot_cross_format_heatmap, plot_actual_vs_predicted,
    plot_residuals, plot_learning_curves, COLORS
)

FORMAT_KEY = 'mens_odi'

## 6.1 Load All Results

In [2]:
# Load test data
train_df, test_df = load_prepared_data(FORMAT_KEY)
X_test, y_test, feature_names = prepare_features(test_df)

# Load models and generate predictions
predictions = {}

# DLS
if 'dls_predicted_final' in test_df.columns:
    predictions['DLS'] = test_df['dls_predicted_final'].fillna(y_test.mean()).values

# ML Models
model_names_map = {
    'xgboost': 'XGBoost',
    'random_forest': 'RandomForest',
    'lightgbm': 'LightGBM',
    'neural_network': 'NeuralNetwork',
}

for file_name, display_name in model_names_map.items():
    try:
        model, scaler = load_model(file_name, format_key=FORMAT_KEY)
        predictions[display_name] = predict_with_model(model, X_test, scaler=scaler)
        print(f"Loaded {display_name}")
    except FileNotFoundError:
        print(f"Not found: {file_name}")

print(f"\nModels loaded: {list(predictions.keys())}")

2026-02-06 17:47:44,006 - INFO - Feature matrix: (25392, 22), Target: (25392,)


Loaded XGBoost
Loaded RandomForest
Loaded LightGBM
Not found: neural_network

Models loaded: ['DLS', 'XGBoost', 'RandomForest', 'LightGBM']


In [3]:
# Run evaluation
eval_results = full_evaluation(test_df, predictions, format_key=FORMAT_KEY, overs_limit=50)

2026-02-06 17:47:45,030 - INFO - 
Overall Results (mens_odi):
2026-02-06 17:47:45,035 - INFO - 
       Model  Subset  RMSE   MAE     R2  MAPE  Within_5  Within_10  Within_20     N
         DLS Overall 61.31 41.18 0.2443 17.59     13.94      25.86      42.93 25392
     XGBoost Overall 47.59 34.97 0.5446 16.38     12.80      24.39      42.78 25392
RandomForest Overall 46.80 34.04 0.5597 15.79     13.87      25.56      44.32 25392
    LightGBM Overall 46.51 34.04 0.5651 15.92     13.24      24.75      43.72 25392
2026-02-06 17:47:45,063 - INFO - 
Phase-wise Results:
2026-02-06 17:47:45,067 - INFO - 
       Model         Subset   RMSE   MAE      R2  MAPE  Within_5  Within_10  Within_20     N
         DLS   Early (1-10) 101.47 78.12 -0.8760 34.29      4.54       9.45      18.34  5420
     XGBoost   Early (1-10)  68.01 54.26  0.1573 26.60      5.98      12.31      23.93  5420
RandomForest   Early (1-10)  67.67 53.95  0.1657 26.30      6.53      12.80      23.91  5420
    LightGBM   Early (1-

## 6.2 Figure 1: DLS Resource Curves

In [4]:
dls = DLSModel(overs_limit=50)
try:
    dls.load()
    print("Loaded fitted DLS model")
except FileNotFoundError:
    print("DLS model not found, using defaults")

plot_dls_resource_curves(dls, format_key=FORMAT_KEY)
print("✓ Figure 1: DLS Resource Curves saved")

2026-02-06 17:47:45,119 - INFO - DLS model loaded from /Users/akshma/Downloads/rajat_thesis/results/models/dls_model_50.pkl


Loaded fitted DLS model


2026-02-06 17:47:45,607 - INFO - Saved: /Users/akshma/Downloads/rajat_thesis/results/figures/mens_odi_dls_resource_curves.png


✓ Figure 1: DLS Resource Curves saved


## 6.3 Figure 2: Model Performance Comparison

In [5]:
plot_model_comparison_bar(eval_results['overall'], format_key=FORMAT_KEY)
print("✓ Figure 2: Model Comparison Bar Chart saved")

2026-02-06 17:47:45,962 - INFO - Saved: /Users/akshma/Downloads/rajat_thesis/results/figures/mens_odi_model_comparison_bar.png


✓ Figure 2: Model Comparison Bar Chart saved


## 6.4 Figure 3: Prediction Error Distributions

In [6]:
plot_error_distributions(y_test.values, predictions, format_key=FORMAT_KEY)
print("✓ Figure 3: Error Distributions saved")

2026-02-06 17:47:46,716 - INFO - Saved: /Users/akshma/Downloads/rajat_thesis/results/figures/mens_odi_error_distributions.png


✓ Figure 3: Error Distributions saved


## 6.5 Figure 4: Phase-wise RMSE

In [7]:
plot_phase_rmse(eval_results['phase_wise'], format_key=FORMAT_KEY)
print("✓ Figure 4: Phase-wise RMSE saved")

2026-02-06 17:47:46,928 - INFO - Saved: /Users/akshma/Downloads/rajat_thesis/results/figures/mens_odi_phase_rmse.png


✓ Figure 4: Phase-wise RMSE saved


## 6.6 Figures 5-6: SHAP & LIME

These are generated in Notebook 05. Verify they exist.

In [8]:
# Check SHAP/LIME figures exist
shap_files = list(FIGURES_DIR.glob('*shap*'))
lime_files = list(FIGURES_DIR.glob('*lime*'))

print(f"SHAP figures found: {len(shap_files)}")
for f in sorted(shap_files)[:5]:
    print(f"  {f.name}")

print(f"\nLIME figures found: {len(lime_files)}")
for f in sorted(lime_files)[:5]:
    print(f"  {f.name}")

print("\n✓ Figures 5-6: SHAP & LIME (from Notebook 05)")

SHAP figures found: 30
  mens_odi_shap_bar_lightgbm.png
  mens_odi_shap_bar_randomforest.png
  mens_odi_shap_bar_xgboost.png
  mens_odi_shap_dep_boundary_percentage_randomforest.png
  mens_odi_shap_dep_current_run_rate_lightgbm.png

LIME figures found: 10
  mens_odi_lime_0_xgboost.html
  mens_odi_lime_0_xgboost.png
  mens_odi_lime_1_xgboost.html
  mens_odi_lime_1_xgboost.png
  mens_odi_lime_2_xgboost.html

✓ Figures 5-6: SHAP & LIME (from Notebook 05)


## 6.7 Figure 7: Cross-Format Heatmap

In [9]:
# Load cross-format data if available
try:
    heatmap_data = pd.read_csv(TABLES_DIR / 'cross_format_rmse.csv', index_col=0)
    plot_cross_format_heatmap(heatmap_data)
    print("✓ Figure 7: Cross-Format Heatmap saved")
except FileNotFoundError:
    print("Cross-format data not available. Run Notebook 04 first.")

Cross-format data not available. Run Notebook 04 first.


## 6.8 Figure 8: Actual vs Predicted

In [10]:
plot_actual_vs_predicted(y_test.values, predictions, format_key=FORMAT_KEY)
print("✓ Figure 8: Actual vs Predicted saved")

2026-02-06 17:47:52,585 - INFO - Saved: /Users/akshma/Downloads/rajat_thesis/results/figures/mens_odi_actual_vs_predicted.png


✓ Figure 8: Actual vs Predicted saved


## 6.9 Figure 9: Residual Plots

In [11]:
plot_residuals(y_test.values, predictions, format_key=FORMAT_KEY)
print("✓ Figure 9: Residual Plots saved")

2026-02-06 17:47:54,702 - INFO - Saved: /Users/akshma/Downloads/rajat_thesis/results/figures/mens_odi_residuals.png


✓ Figure 9: Residual Plots saved


## 6.10 Figure 10: Learning Curves

In [12]:
# Note: Learning curves require the training history which is in memory during training
# This cell creates a placeholder if history is not available
try:
    # Try to load from training results
    from src.visualizations import plot_learning_curves
    print("Learning curves need to be generated during training (Notebook 03).")
    print("Check if the figure exists:")
    lc_path = FIGURES_DIR / f'{FORMAT_KEY}_learning_curves.png'
    print(f"  {lc_path.name}: {'EXISTS' if lc_path.exists() else 'NOT FOUND'}")
except Exception as e:
    print(f"Note: {e}")

print("✓ Figure 10: Learning Curves")

Learning curves need to be generated during training (Notebook 03).
Check if the figure exists:
  mens_odi_learning_curves.png: NOT FOUND
✓ Figure 10: Learning Curves


## 6.11 LaTeX Tables

In [13]:
# Table 1: Overall comparison
latex_overall = generate_latex_table(
    eval_results['overall'],
    caption="Overall Model Comparison on Men's ODI Test Set",
    label="tab:overall_comparison"
)
print("TABLE 1: Overall Comparison")
print(latex_overall)

with open(TABLES_DIR / f'{FORMAT_KEY}_overall_latex.tex', 'w') as f:
    f.write(latex_overall)

TABLE 1: Overall Comparison
\begin{table}[htbp]
\centering
\caption{Overall Model Comparison on Men's ODI Test Set}
\label{tab:overall_comparison}
\begin{tabular}{lrrrrrrrrr}
\toprule
Model & Subset & RMSE & MAE & R2 & MAPE & Within_5 & Within_10 & Within_20 & N \\
\midrule
DLS & Overall & 61.31 & 41.18 & 0.24 & 17.59 & 13.94 & 25.86 & 42.93 & 25392 \\
XGBoost & Overall & 47.59 & 34.97 & 0.54 & 16.38 & 12.80 & 24.39 & 42.78 & 25392 \\
RandomForest & Overall & 46.80 & 34.04 & 0.56 & 15.79 & 13.87 & 25.56 & 44.32 & 25392 \\
LightGBM & Overall & 46.51 & 34.04 & 0.57 & 15.92 & 13.24 & 24.75 & 43.72 & 25392 \\
\bottomrule
\end{tabular}
\end{table}


In [14]:
# Table 2: Phase-wise comparison
latex_phase = generate_latex_table(
    eval_results['phase_wise'],
    caption="Phase-wise Model Comparison on Men's ODI Test Set",
    label="tab:phase_comparison"
)
print("\nTABLE 2: Phase-wise Comparison")
print(latex_phase)

with open(TABLES_DIR / f'{FORMAT_KEY}_phase_latex.tex', 'w') as f:
    f.write(latex_phase)


TABLE 2: Phase-wise Comparison
\begin{table}[htbp]
\centering
\caption{Phase-wise Model Comparison on Men's ODI Test Set}
\label{tab:phase_comparison}
\begin{tabular}{lrrrrrrrrr}
\toprule
Model & Subset & RMSE & MAE & R2 & MAPE & Within_5 & Within_10 & Within_20 & N \\
\midrule
DLS & Early (1-10) & 101.47 & 78.12 & -0.88 & 34.29 & 4.54 & 9.45 & 18.34 & 5420 \\
XGBoost & Early (1-10) & 68.01 & 54.26 & 0.16 & 26.60 & 5.98 & 12.31 & 23.93 & 5420 \\
RandomForest & Early (1-10) & 67.67 & 53.95 & 0.17 & 26.30 & 6.53 & 12.80 & 23.91 & 5420 \\
LightGBM & Early (1-10) & 66.97 & 53.40 & 0.18 & 26.22 & 6.38 & 12.25 & 24.30 & 5420 \\
DLS & Middle (11-30) & 56.56 & 43.32 & 0.40 & 18.48 & 9.11 & 17.57 & 32.71 & 10757 \\
XGBoost & Middle (11-30) & 49.76 & 39.32 & 0.54 & 18.72 & 8.56 & 17.28 & 33.29 & 10757 \\
RandomForest & Middle (11-30) & 48.74 & 38.26 & 0.56 & 17.97 & 8.92 & 17.57 & 35.07 & 10757 \\
LightGBM & Middle (11-30) & 48.48 & 38.23 & 0.56 & 18.15 & 8.86 & 17.83 & 33.91 & 10757 \\
DLS & L

In [15]:
# Table 3: Wicket-state comparison
latex_wicket = generate_latex_table(
    eval_results['wicket_state'],
    caption="Wicket-state Model Comparison on Men's ODI Test Set",
    label="tab:wicket_comparison"
)
print("\nTABLE 3: Wicket-state Comparison")
print(latex_wicket)

with open(TABLES_DIR / f'{FORMAT_KEY}_wicket_latex.tex', 'w') as f:
    f.write(latex_wicket)


TABLE 3: Wicket-state Comparison
\begin{table}[htbp]
\centering
\caption{Wicket-state Model Comparison on Men's ODI Test Set}
\label{tab:wicket_comparison}
\begin{tabular}{lrrrrrrrrr}
\toprule
Model & Subset & RMSE & MAE & R2 & MAPE & Within_5 & Within_10 & Within_20 & N \\
\midrule
DLS & 0-2 wickets & 76.37 & 55.11 & -0.15 & 22.88 & 7.66 & 15.48 & 29.37 & 11468 \\
XGBoost & 0-2 wickets & 57.86 & 44.51 & 0.34 & 19.58 & 8.31 & 16.77 & 32.29 & 11468 \\
RandomForest & 0-2 wickets & 57.27 & 43.94 & 0.35 & 19.34 & 8.39 & 16.90 & 32.77 & 11468 \\
LightGBM & 0-2 wickets & 56.69 & 43.51 & 0.36 & 19.19 & 8.48 & 16.86 & 32.89 & 11468 \\
DLS & 3-5 wickets & 52.33 & 36.31 & 0.37 & 15.69 & 14.30 & 25.85 & 44.55 & 9148 \\
XGBoost & 3-5 wickets & 41.10 & 30.76 & 0.61 & 14.68 & 13.34 & 25.79 & 45.78 & 9148 \\
RandomForest & 3-5 wickets & 40.45 & 30.04 & 0.62 & 14.28 & 14.20 & 26.69 & 47.33 & 9148 \\
LightGBM & 3-5 wickets & 40.30 & 30.14 & 0.63 & 14.41 & 13.52 & 25.91 & 46.40 & 9148 \\
DLS & 6-8 wick

## 6.12 Final Summary

In [16]:
print("="*60)
print("THESIS RESULTS SUMMARY")
print("="*60)

overall = eval_results['overall']
print("\n1. OVERALL PERFORMANCE")
print(overall.to_string(index=False))

# Best ML model
ml_models = overall[overall['Model'] != 'DLS']
if not ml_models.empty:
    best = ml_models.loc[ml_models['RMSE'].idxmin()]
    print(f"\nBest ML Model: {best['Model']}")
    print(f"  RMSE: {best['RMSE']:.2f}")
    print(f"  MAE: {best['MAE']:.2f}")
    print(f"  R²: {best['R2']:.4f}")

    dls_row = overall[overall['Model'] == 'DLS']
    if not dls_row.empty:
        dls_rmse = dls_row.iloc[0]['RMSE']
        improvement = ((dls_rmse - best['RMSE']) / dls_rmse) * 100
        print(f"\n  Improvement over DLS: {improvement:.1f}% RMSE reduction")

# File counts
figures = list(FIGURES_DIR.glob('*.png'))
tables = list(TABLES_DIR.glob('*.csv'))
latex = list(TABLES_DIR.glob('*.tex'))

print(f"\n2. OUTPUT FILES")
print(f"  Figures (PNG): {len(figures)}")
print(f"  Tables (CSV): {len(tables)}")
print(f"  LaTeX tables: {len(latex)}")

print("\n" + "="*60)
print("✓ All visualizations and results generated!")
print("="*60)

THESIS RESULTS SUMMARY

1. OVERALL PERFORMANCE
       Model  Subset  RMSE   MAE     R2  MAPE  Within_5  Within_10  Within_20     N
         DLS Overall 61.31 41.18 0.2443 17.59     13.94      25.86      42.93 25392
     XGBoost Overall 47.59 34.97 0.5446 16.38     12.80      24.39      42.78 25392
RandomForest Overall 46.80 34.04 0.5597 15.79     13.87      25.56      44.32 25392
    LightGBM Overall 46.51 34.04 0.5651 15.92     13.24      24.75      43.72 25392

Best ML Model: LightGBM
  RMSE: 46.51
  MAE: 34.04
  R²: 0.5651

  Improvement over DLS: 24.1% RMSE reduction

2. OUTPUT FILES
  Figures (PNG): 43
  Tables (CSV): 3
  LaTeX tables: 3

✓ All visualizations and results generated!
